# Week 10 — Day 2 : MCP Client with an LLM

**Bootcamp GenAI & Machine Learning 2026 — Developers Institute**

Build an MCP client that connects to a math server over STDIO, converts tools to LLM function specs, and uses a stub (or real) LLM to plan and execute tool calls.

| Mode | Config |
|------|--------|
| **Stub** (default) | No token needed — hardcoded planner simulates LLM |
| **Real LLM** | Set `GITHUB_TOKEN` env var to use a real model |

---
## Setup

In [ ]:
%pip install -q "mcp[cli]" openai

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("CalcDemo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Returns the sum of two integers."""
    return a + b


@mcp.resource("ops://list")
def list_ops() -> str:
    """Returns the list of available math operations."""
    return "add"


if __name__ == "__main__":
    mcp.run()

In [ ]:
import os

# Toggle: set USE_REAL_LLM = True and provide GITHUB_TOKEN for a real model
USE_REAL_LLM = False
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

print(f"Mode: {'Real LLM' if USE_REAL_LLM and GITHUB_TOKEN else 'Stub (no token needed)'}")

---
## Exercise 1 — Theory: Why is STDIO transport simpler than HTTP for local MCP dev?

**Answer:**

| Aspect | STDIO | HTTP |
|--------|-------|------|
| Setup | Zero config — server is a subprocess | Requires binding a port, handling addresses |
| Auth | Inherited from parent process | Needs tokens, headers, TLS |
| Lifecycle | Server starts/stops with client | Server must be managed separately |
| Debugging | Logs go to stderr, visible in terminal | Needs separate log tailing |
| Port conflicts | None — no port used | Can collide with other services |

**Summary:** STDIO is simpler because the client just spawns the server as a child process and communicates via stdin/stdout — no network configuration, no authentication, and the server's lifetime is automatically tied to the client's lifetime. HTTP becomes necessary only when the server needs to be remote or shared across multiple clients.

---
## Exercise 2 — Connect & Initialize

In [ ]:
import asyncio
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Server parameters — spawns server.py via mcp CLI over STDIO
server_params = StdioServerParameters(
    command="mcp", args=["run", "server.py"], env=None
)

print("Server params configured ✅")
print(f"  command : {server_params.command}")
print(f"  args    : {server_params.args}")

In [ ]:
async def connect_and_init():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize the MCP session (capability negotiation)
            result = await session.initialize()
            print("Session initialized ✅")
            print(f"  Server name    : {result.serverInfo.name}")
            print(f"  Protocol version: {result.protocolVersion}")

asyncio.run(connect_and_init())

---
## Exercise 3 — Discover: List Resources and Tools

In [ ]:
async def discover():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # --- Resources ---
            resources = await session.list_resources()
            print("=== Resources ===")
            for r in resources.resources:
                print(f"  name : {r.name}")
                print(f"  uri  : {r.uri}")
                print()

            # --- Tools ---
            tools = await session.list_tools()
            print("=== Tools ===")
            for t in tools.tools:
                print(f"  name        : {t.name}")
                print(f"  description : {t.description}")
                print(f"  inputSchema :")
                # Print each property in the input schema
                props = t.inputSchema.get("properties", {})
                required = t.inputSchema.get("required", [])
                for prop_name, prop_info in props.items():
                    req = "required" if prop_name in required else "optional"
                    print(f"    - {prop_name} ({prop_info.get('type', '?')}) [{req}]")
                print()

asyncio.run(discover())

---
## Exercise 4 — Convert MCP Tools to LLM Function Specs

LLMs (OpenAI, GitHub Models) expect tools in a specific JSON format. We need to convert MCP tool objects into that format.

In [ ]:
def convert_to_llm_tool(mcp_tool) -> dict:
    """
    Convert an MCP tool object into an OpenAI-compatible function spec.

    MCP tool has:
      .name        → str
      .description → str | None
      .inputSchema → dict (JSON Schema)

    LLM expects:
      {"type": "function", "function": {"name": ..., "description": ..., "parameters": ...}}
    """
    return {
        "type": "function",
        "function": {
            "name": mcp_tool.name,
            "description": mcp_tool.description or "",
            "parameters": mcp_tool.inputSchema,
        }
    }


async def build_function_list():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()

            llm_tools = [convert_to_llm_tool(t) for t in tools.tools]

            print("=== LLM Function Specs ===")
            for spec in llm_tools:
                print(json.dumps(spec, indent=2))
                print()

            return llm_tools

asyncio.run(build_function_list())

---
## Exercise 5 — Plan & Execute with Stub (or Real) LLM

**Prompt:** `"Add 2 to 20."`

The planner decides which tool to call and with which arguments. The executor calls the MCP server.

In [ ]:
# --- Stub Planner (no token needed) ---
def stub_planner(prompt: str, llm_tools: list) -> list:
    """
    Simulates an LLM that parses the prompt and returns tool_calls.
    In stub mode, we hardcode the response for known prompts.
    """
    prompt_lower = prompt.lower()

    # Detect "add X to Y" pattern
    if "add" in prompt_lower:
        import re
        nums = list(map(int, re.findall(r'\d+', prompt)))
        a, b = (nums[0], nums[1]) if len(nums) >= 2 else (2, 20)
        return [{"name": "add", "arguments": {"a": a, "b": b}}]

    if "multiply" in prompt_lower:
        import re
        nums = list(map(int, re.findall(r'\d+', prompt)))
        a, b = (nums[0], nums[1]) if len(nums) >= 2 else (3, 7)
        return [{"name": "multiply", "arguments": {"a": a, "b": b}}]

    return []  # No tool call identified


# --- Real LLM Planner (requires GITHUB_TOKEN) ---
def real_llm_planner(prompt: str, llm_tools: list) -> list:
    from openai import OpenAI
    client = OpenAI(
        base_url="https://models.inference.ai.azure.com",
        api_key=GITHUB_TOKEN
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        tools=llm_tools,
        tool_choice="auto"
    )
    msg = response.choices[0].message
    if msg.tool_calls:
        return [
            {"name": tc.function.name, "arguments": json.loads(tc.function.arguments)}
            for tc in msg.tool_calls
        ]
    return []


def planner(prompt: str, llm_tools: list) -> list:
    if USE_REAL_LLM and GITHUB_TOKEN:
        return real_llm_planner(prompt, llm_tools)
    return stub_planner(prompt, llm_tools)


print("Planner defined ✅")

In [ ]:
# --- Executor ---
async def execute_tool_calls(tool_calls: list, session: ClientSession) -> list:
    """Execute a list of tool_calls against the MCP server and return results."""
    results = []
    for tc in tool_calls:
        result = await session.call_tool(tc["name"], tc["arguments"])
        results.append({
            "tool": tc["name"],
            "arguments": tc["arguments"],
            "result": result.content[0].text
        })
    return results


# --- Full plan-and-execute loop ---
async def plan_and_execute(prompt: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 1. Get LLM function specs
            tools = await session.list_tools()
            llm_tools = [convert_to_llm_tool(t) for t in tools.tools]

            print(f"Prompt  : {prompt!r}")
            print(f"Tools   : {[t['function']['name'] for t in llm_tools]}")
            print()

            # 2. Plan — LLM (or stub) proposes tool calls
            tool_calls = planner(prompt, llm_tools)
            print("=== Planned tool calls ===")
            for tc in tool_calls:
                print(f"  {tc['name']}({tc['arguments']})")
            print()

            # 3. Execute — call the MCP server
            results = await execute_tool_calls(tool_calls, session)
            print("=== Execution results ===")
            for r in results:
                print(f"  {r['tool']}({r['arguments']}) → {r['result']}")


asyncio.run(plan_and_execute("Add 2 to 20."))

---
## Optional — Add `multiply(a, b)` Tool

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("CalcDemo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Returns the sum of two integers."""
    return a + b


@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Returns the product of two integers."""
    return a * b


@mcp.resource("ops://list")
def list_ops() -> str:
    """Returns the list of available math operations."""
    return "add\nmultiply"


if __name__ == "__main__":
    mcp.run()

In [ ]:
# Rebuild function list and rerun planner with multiply
print("=== With multiply tool added ===")
asyncio.run(plan_and_execute("Add 2 to 20."))
print()
asyncio.run(plan_and_execute("Multiply 3 by 7."))

---
## Summary

| Exercise | What was built | Status |
|----------|---------------|--------|
| 1 | Theory — STDIO vs HTTP for local MCP dev | ✅ |
| 2 | ClientSession initialization over STDIO | ✅ |
| 3 | Discover resources + tools + inputSchema properties | ✅ |
| 4 | `convert_to_llm_tool` — MCP → OpenAI function spec | ✅ |
| 5 | Stub planner + executor — `"Add 2 to 20."` → `add(2,20)` → `22` | ✅ |
| Optional | `multiply` tool added, function list rebuilt, both prompts work | ✅ |

### Key Insight — The MCP + LLM pattern

```
User prompt
    └──► Planner (LLM)  ←── function specs (from convert_to_llm_tool)
              │
              ▼ tool_calls [{name, arguments}]
         Executor
              │
              ▼ session.call_tool(name, args)
         MCP Server  →  result
```

The LLM never touches the MCP server directly — it just proposes tool calls in a structured format. The executor bridges the gap, translating LLM output into MCP calls and returning results.